# Nhận dạng chữ tiếng Việt — VietOCR fine-tune (Kaggle)

**Trước khi chạy, bật trong Notebook settings:**

| Cài đặt | Giá trị |
|---|---|
| Accelerator | **GPU P100** — VietOCR `Trainer` chỉ chạy 1 GPU (`config['device']` đơn lẻ, không có DataParallel). T4 x2 sẽ phí mất một con, và một T4 chậm hơn P100 với VGG19-bn. |
| Internet | **On** (bắt buộc — cài `vietocr`, tải config và trọng số pretrained) |
| Persistence | Files only |

**Dữ liệu:** *Add Data → Upload* `vi_rec_100k.zip` thành Kaggle Dataset. Kaggle tự giải nén vào `/kaggle/input/` nên không tốn hạn mức 20GB của `/kaggle/working`. Không có thì script tự `gdown`.

---
### Vì sao đổi từ PaddleOCR sang VietOCR

Bản PaddleOCR (PP-OCRv5 mobile rec, CTC) hội tụ ở `acc = 0.606`, `NED = 0.9709`. Phân tích lỗi cho thấy phần lớn là **sai đúng 1 ký tự dấu** (`củi`→`cúi`, `đồng`→`dòng`), cộng một nhóm dòng **bị cụt đuôi**. Cả hai đều là điểm yếu cố hữu của CTC + greedy decode: giải mã từng frame độc lập không có ngữ cảnh, và ràng buộc `T ≥ L` khiến nhãn dài bị cắt.

VietOCR dùng **decoder tự hồi quy** — mỗi ký tự sinh ra có điều kiện trên các ký tự trước — nên có mô hình ngôn ngữ ngầm sửa đúng loại lỗi này, và không có ràng buộc độ dài nào.

### Hai sửa đổi so với cấu hình VietOCR mặc định

1. **`image_height` 32 → 64.** Ở h=32, nét gạch ngang của `đ` và dấu mũ chỉ dày 1–2 pixel nên hay bị mất. Kèm theo **bắt buộc** sửa `cnn.ss`/`cnn.ks` hàng cuối `[1,1]` → `[2,1]`: tích stride chiều cao thành 32, nhờ vậy feature height vẫn bằng 2 và chuỗi encoder **không dài thêm** (nếu không sẽ gấp đôi, attention nặng gấp 4). Lớp pooling không có tham số nên đổi `ss`/`ks` không làm lệch trọng số pretrained.
2. **`image_max_width` 512 → 1024.** Tỉ lệ w/h trung vị là 19.7; ở h=64 ảnh trung vị cần ~1260px. Để 512 là nén nát gần hết ảnh.

### Về ngân sách 12h

Cell 6 **đo tốc độ thật** (200 iter train + 100 ảnh suy luận) rồi in bảng dự toán và tự chọn số epoch vừa ngân sách. Notebook luôn chạy hết mọi cell trong một phiên — số epoch bị co lại chứ không để Kaggle cắt ngang.


## 1. Cấu hình


In [ ]:
# ============================================================================
# CAU HINH — chinh o day, khong sua o duoi
# ============================================================================
import time
SESSION_START = time.time()

SEED             = 2026
IGNORE_SPACE     = True    # dinh nghia acc; phai ghi ro trong bao cao

# --- anh dau vao ---
IMG_HEIGHT       = 64      # mac dinh VietOCR la 32; nang len de giu duoc dau thanh + net gach cua d
IMG_MIN_WIDTH    = 32
IMG_MAX_WIDTH    = 1024    # mac dinh 512 qua nho: ty le w/h trung vi 19.7 -> o h=64 can ~1260px
# Tich stride chieu cao cua CNN phai la 32 (khong phai 16) de h=64 van cho
# feature height = 2, tuc chuoi encoder dai y het cau hinh goc.
# Lop pooling khong co tham so nen doi ss/ks KHONG lam lech trong so pretrained.
CNN_SS           = [[2, 2], [2, 2], [2, 1], [2, 1], [2, 1]]
CNN_KS           = [[2, 2], [2, 2], [2, 1], [2, 1], [2, 1]]

# --- muc 2.2 fine-tune ---
TARGET_EPOCHS    = 16      # so epoch mong muon; se bi co lai neu khong du ngan sach
BATCH_SIZE       = 32
MAX_LR           = 3e-4    # AdamW + OneCycleLR
PCT_START        = 0.1
EVAL_PER_EPOCH   = 2       # so lan do tren val moi epoch -> so diem tren duong cong

# --- muc 2.4 ablation: so epoch 2 vs 4, doc tu chinh lan train nay ---
ABLATION_EPOCHS  = [2, 4]

# --- hieu chuan thoi gian ---
CALIB_ITERS      = 200     # so iter chay thu de do giay/iter
CALIB_IMAGES     = 100     # so anh chay thu de do giay/anh khi suy luan
RESERVE_MIN      = 20      # phut du phong ngoai cac chi phi da do duoc

# --- ngan sach thoi gian (Kaggle cat o 12h) ---
SESSION_BUDGET_H = 11.0

# --- nguon ---
VIETOCR_PIP      = "vietocr"
MODEL_NAME       = "vgg_transformer"   # hoac vgg_seq2seq
DRIVE_ID         = "1_NKW1CL49NKtnT92ddaNZwGcaFJOkUgM"   # chi dung neu khong add dataset

def time_left():
    return SESSION_BUDGET_H * 3600 - (time.time() - SESSION_START)

def hms(s):
    s = int(max(s, 0)); return f"{s//3600}h{(s%3600)//60:02d}m{s%60:02d}s"

print(f"Ngan sach phien: {SESSION_BUDGET_H}h  |  con lai {hms(time_left())}")
print(f"Anh: {IMG_HEIGHT} x [{IMG_MIN_WIDTH}..{IMG_MAX_WIDTH}]")
print(f"Muc tieu: {TARGET_EPOCHS} epoch (se co lai neu khong du gio)")


## 2. Cài đặt

Đề lưu ý: `pip install` trên Kaggle có thể thất bại âm thầm — cell này xác nhận bằng `pip show` trước khi bạn Save & Run All.


In [ ]:
# ============================================================================
# CAI DAT — de bai luu y: !pip install tren Kaggle co the that bai AM THAM,
# nen phai xac nhan bang pip show truoc khi Save & Run All.
# ============================================================================
import subprocess, sys, importlib

def sh(*args, check=True):
    p = subprocess.run([str(x) for x in args], capture_output=True, text=True)
    if check and p.returncode != 0:
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise RuntimeError("that bai: " + " ".join(map(str, args)))
    return p.stdout

try:
    importlib.import_module("vietocr")
    print("vietocr da co san")
except ImportError:
    print("dang cai vietocr ...")
    sh(sys.executable, "-m", "pip", "install", "-q", VIETOCR_PIP)

for pkg in ["rapidfuzz", "gdown"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        sh(sys.executable, "-m", "pip", "install", "-q", pkg)

# ---------------------------------------------------------------------------
# Ban vietocr tren PyPI import `imgaug`, ma imgaug 0.4.0 (2020) dung np.sctypes —
# API da bi XOA trong NumPy 2.0, con Kaggle nay ship NumPy 2.x. Ket qua: Predictor
# chay binh thuong (khong dung aug.py) nhung Trainer chet ngay luc import.
#
# Hai duong sua, thu lan luot:
#   1) shim lai np.sctypes truoc khi import imgaug  (da kiem chung tren numpy 2.0.2)
#   2) cai vietocr tu GitHub master — ban do da bo imgaug, chuyen sang albumentations
# ---------------------------------------------------------------------------
NP2_SHIM = '''import numpy as _np
if not hasattr(_np, "sctypes"):
    _np.sctypes = {"int": [_np.int8, _np.int16, _np.int32, _np.int64],
                   "uint": [_np.uint8, _np.uint16, _np.uint32, _np.uint64],
                   "float": [_np.float16, _np.float32, _np.float64],
                   "complex": [_np.complex64, _np.complex128],
                   "others": [bool, object, bytes, str, _np.void]}
for _n, _v in [("bool", bool), ("int", int), ("float", float), ("complex", complex),
               ("object", object), ("str", str), ("unicode", str)]:
    if _n not in _np.__dict__:        # tranh __getattr__ cua numpy 2 (FutureWarning)
        setattr(_np, _n, _v)
'''

def trainer_imports():
    p = subprocess.run([sys.executable, "-c", NP2_SHIM +
                        "from vietocr.model.trainer import Trainer; print('TRAINER_OK')"],
                       capture_output=True, text=True)
    return "TRAINER_OK" in p.stdout, (p.stderr or p.stdout)[-1500:]

ok, err = trainer_imports()
if ok:
    print("\nTrainer import duoc (kem shim numpy)")
else:
    print("\nTrainer KHONG import duoc voi ban PyPI:")
    print("  ", err.strip().splitlines()[-1] if err.strip() else "(khong ro)")
    print("  -> cai vietocr tu GitHub master (ban do dung albumentations, bo imgaug)")
    sh(sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
       "--no-deps", "git+https://github.com/pbcquoc/vietocr.git")
    sh(sys.executable, "-m", "pip", "install", "-q", "albumentations")
    ok, err = trainer_imports()
    assert ok, ("Van khong import duoc vietocr.model.trainer:\n" + err +
                "\n\nThu them: !pip install -q 'numpy<2' roi Restart & Run All.")
    print("  OK sau khi cai tu GitHub")

# --- xac nhan that su cai duoc, khong tin vao ma thoat cua pip ---
for pkg in ["vietocr", "torch", "rapidfuzz"]:
    out = sh(sys.executable, "-m", "pip", "show", pkg, check=False)
    ver = next((l.split(": ", 1)[1] for l in out.splitlines() if l.startswith("Version:")), None)
    assert ver, f"{pkg} KHONG cai duoc — dung lai, dung Save & Run All"
    print(f"  {pkg:12s} {ver}")

# --- GPU ---
info = subprocess.run(
    [sys.executable, "-c",
     "import torch;print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(),"
     "torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"],
    capture_output=True, text=True).stdout.strip()
print("\ntorch:", info)
assert "True" in info, "Khong thay GPU — bat Accelerator trong Notebook settings"

N_GPU = int(info.split()[2])
if N_GPU > 1:
    print(f"\nLuu y: thay {N_GPU} GPU nhung VietOCR Trainer chi chay tren 1 GPU\n"
          f"       (config['device'] don le, khong co DataParallel).\n"
          f"       Nen doi Accelerator sang P100 — bang thong bo nho 732 GB/s so voi\n"
          f"       320 GB/s cua T4, nhanh hon ro voi VGG19-bn.")


## 3. Đường dẫn và dữ liệu


In [ ]:
# ============================================================================
# DUONG DAN + TIM DU LIEU
#   Uu tien: add vi_rec_100k.zip lam Kaggle Dataset (Kaggle tu giai nen,
#   nam o /kaggle/input/... -> khong ton dung luong /kaggle/working 20GB).
#   Neu khong co -> tu tai bang gdown ve /kaggle/temp.
# ============================================================================
from pathlib import Path
import os, shutil, zipfile, subprocess, sys

ON_KAGGLE = Path("/kaggle").exists()
WORK    = Path("/kaggle/working/vi_ocr") if ON_KAGGLE else Path.cwd() / "vi_ocr"
SCRATCH = Path("/kaggle/temp") if ON_KAGGLE else WORK / "temp"

CONFIG_DIR, OUTPUT_DIR, RESULTS_DIR = WORK / "configs", WORK / "output", WORK / "results"
LOG_DIR, SCRIPT_DIR = WORK / "logs", WORK / "scripts"
LMDB_DIR = SCRATCH / "lmdb"          # LMDB rat nang -> KHONG de trong /kaggle/working
for p in [WORK, SCRATCH, CONFIG_DIR, OUTPUT_DIR, RESULTS_DIR, LOG_DIR, SCRIPT_DIR, LMDB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv"], capture_output=True, text=True).stdout)

# vi_dict.txt KHONG nam trong NEED: no hay duoc de rieng o goc dataset thay vi
# trong zip. Tim rieng roi copy vao sau.
NEED = ["rec_val.txt", "rec_test.txt", "train", "val", "test"]

def find_dataset_root(*roots):
    """Tra ve (thu_muc_dung, danh_sach_gan_dung) — phan thu hai de bao loi cho ro."""
    cands, near = [], []
    for root in roots:
        if not Path(root).exists():
            continue
        for f in Path(root).rglob("rec_train.txt"):
            d = f.parent
            if "__MACOSX" in d.parts:
                continue
            missing = [n for n in NEED if not (d / n).exists()]
            (cands if not missing else near).append((d, missing))
    cands.sort(key=lambda t: len(t[0].parts))
    return (cands[0][0] if cands else None), near


def unzip_all(zp, dest):
    """Giai nen zp vao dest, roi giai nen tiep cac zip long ben trong.
    /kaggle/temp khong duoc luu giua cac phien nen buoc nay lap lai moi lan —
    khoang vai phut, da nam trong ngan sach vi SESSION_START dat o cell 1."""
    t0 = time.time()
    print(f"  giai nen {zp.name} ({zp.stat().st_size/1e9:.2f} GB) -> {dest} ...")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(dest)
    for nested in list(dest.rglob("*.zip")):
        if "__MACOSX" in nested.parts or nested.name.startswith("._"):
            continue
        print(f"  giai nen long {nested.name} ...")
        with zipfile.ZipFile(nested) as zf:
            zf.extractall(nested.parent)
    print(f"  xong sau {hms(time.time() - t0)}  |  ngan sach con {hms(time_left())}")


DATA_DIR, near_miss = find_dataset_root("/kaggle/input", SCRATCH, WORK)

# Kaggle KHONG phai luc nao cung tu giai nen: neu dataset chi chua file .zip thi
# phai tu bung ra. /kaggle/input chi doc nen bung vao SCRATCH.
if DATA_DIR is None and Path("/kaggle/input").exists():
    # Khong loc theo kich thuoc: chi lay zip lon nhat. Loc theo nguong se hong
    # neu ban nen ky hon hoac chia nho file.
    zips = [z for z in Path("/kaggle/input").rglob("*.zip")
            if "__MACOSX" not in z.parts and not z.name.startswith("._")]
    if zips:
        print("Thay file zip chua giai nen trong /kaggle/input:")
        for z in zips:
            print("  ", z)
        unzip_all(max(zips, key=lambda z: z.stat().st_size), SCRATCH / "data")
        DATA_DIR, near_miss = find_dataset_root(SCRATCH)

if DATA_DIR is None:
    # In ra /kaggle/input that su co gi — 90% truong hop la quen Add Data
    print("KHONG thay dataset. /kaggle/input dang co:")
    inp = Path("/kaggle/input")
    if not inp.exists() or not any(inp.iterdir()):
        print("   (TRONG RONG — chua Add Data vao notebook nay)")
    else:
        for d in sorted(inp.iterdir()):
            print("  ", d.name)
            for s in sorted(d.iterdir())[:8] if d.is_dir() else []:
                print("      ", s.name + ("/" if s.is_dir() else ""))
    for d, missing in near_miss:
        print(f"\n   gan dung: {d}\n     thieu: {missing}")

    print("\nCACH SUA (nen lam):\n"
          "  Add Data -> chon dataset vi_rec_100k ban da upload tu truoc.\n"
          "  Notebook MOI khong tu keo Add Data tu notebook cu sang.\n")

    print("Khong co thi tai bang gdown (~3.8GB) ...")
    import gdown
    zp = SCRATCH / "vi_rec_100k.zip"
    if not zp.exists():
        try:
            gdown.download(id=DRIVE_ID, output=str(zp), quiet=False)
        except Exception as e:
            print("gdown loi:", e)
    # gdown that bai tra ve None chu KHONG raise -> phai tu kiem tra
    assert zp.exists() and zp.stat().st_size > 10 ** 8, (
        "\n\nTai bang gdown that bai.\n"
        "Neu Google bao 'Cannot retrieve the public link ... or have had many accesses'\n"
        "thi do la HAN NGACH TAI cua Google Drive, khong phai quyen truy cap — doi\n"
        "quyen chia se khong giai quyet duoc. Quota thuong mo lai sau ~24h.\n\n"
        "Cach dung: tai zip ve may mot lan, roi Kaggle -> Datasets -> New Dataset\n"
        "-> upload, sau do Add Data vao notebook nay. Kaggle tu giai nen, khong ton\n"
        "han muc 20GB cua /kaggle/working, va khong bao gio dinh quota nua.")
    unzip_all(zp, SCRATCH / "data")
    DATA_DIR, near_miss = find_dataset_root(SCRATCH)

assert DATA_DIR is not None, (
    "Giai nen xong nhung van khong thay du rec_train.txt / rec_val.txt / rec_test.txt /"
    f" train/ / val/ / test/ trong cung mot thu muc.\nGan dung: {near_miss}")
TRAIN_FILE, VAL_FILE = DATA_DIR / "rec_train.txt", DATA_DIR / "rec_val.txt"
TEST_FILE = DATA_DIR / "rec_test.txt"

# vi_dict.txt hay duoc de RIENG o goc dataset (ngoai zip). Tim khap /kaggle/input
# roi copy vao thu muc lam viec — DATA_DIR co the nam trong /kaggle/input (chi doc).
DICT_FILE = DATA_DIR / "vi_dict.txt"
if not DICT_FILE.exists():
    cands = []
    for root in [Path("/kaggle/input"), SCRATCH, WORK]:
        if root.exists():
            cands += [p for p in root.rglob("vi_dict.txt") if "__MACOSX" not in p.parts]
    assert cands, (
        "Khong thay vi_dict.txt o bat ky dau trong /kaggle/input hay /kaggle/temp.\n"
        "No phai nam cung thu muc voi rec_train.txt, hoac de rieng o goc dataset.")
    DICT_FILE = WORK / "vi_dict.txt"
    shutil.copy(cands[0], DICT_FILE)
    print("vi_dict.txt lay tu:", cands[0])

# ---------------------------------------------------------------------------
# CHECKPOINT PHIEN TRUOC (add output notebook cu / Kaggle Model de train tiep)
#
# Ba cai bay da gap, cell nay xu ly ca ba:
#   1. Ten file khong doan truoc duoc: Kaggle Model giu nguyen ten ban upload
#      (vietocr_checkpoint.zip, .pth, hoac khong duoi). Chi tim ".pth" la truot.
#   2. Zip bi bung phang: file .pth cua torch VON LA zip voi moi entry nam trong
#      thu muc con ("archive/data.pkl"). Neu bi giai nen roi nen lai, entry roi
#      ra goc -> torch.load bao:
#         "file in archive is not in a subdirectory: byteorder"
#      -> boc lai vao archive/ la chay duoc, du lieu ben trong khong hong.
#   3. Kaggle giai nen han thanh THU MUC (data.pkl, byteorder, data/ nam roi)
#      -> nen lai thanh mot file .pth dung chuan.
#
# Khong tim thay thi RESUME_CKPT = None va train tu dau, KHONG de duong dan chet
# roi chet o cell train.
# ---------------------------------------------------------------------------
import zipfile

def _repair_flat_zip(src):
    """Boc lai zip checkpoint neu entry bi nam o goc. Tra ve duong dan dung duoc."""
    src = Path(src)
    if not zipfile.is_zipfile(src):
        return src
    with zipfile.ZipFile(src) as z:
        names = [n for n in z.namelist() if not n.endswith("/")]
    if not any("/" not in n for n in names):
        return src                                  # cau truc da dung
    fixed = SCRATCH / "resume_repaired.pth"
    with zipfile.ZipFile(src) as zin, zipfile.ZipFile(fixed, "w", zipfile.ZIP_STORED) as zout:
        for info in zin.infolist():
            if not info.is_dir():
                zout.writestr("archive/" + info.filename, zin.read(info.filename))
    print("  [va] entry nam o goc zip -> boc lai vao archive/:", fixed)
    return fixed


def _rezip_extracted_dir(d):
    """Kaggle da giai nen .pth thanh thu muc -> nen lai thanh file .pth dung chuan."""
    d = Path(d)
    if not (d / "data.pkl").exists():
        return None
    out = SCRATCH / (d.name + "_rezip.pth")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for f in sorted(d.rglob("*")):
            if f.is_file():
                z.write(f, "archive/" + str(f.relative_to(d)))
    print("  [va] thu muc da giai nen -> nen lai:", out)
    return out


def _usable_checkpoint(path):
    """None neu khong nap duoc hoac khong phai checkpoint train co optimizer."""
    import torch
    try:
        path = _repair_flat_zip(path)
        obj = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print("  [bo qua]", path, "->", str(e)[:120])
        return None
    if not (isinstance(obj, dict) and {"state_dict", "optimizer", "iter"} <= set(obj)):
        keys = list(obj)[:6] if isinstance(obj, dict) else type(obj).__name__
        print("  [bo qua]", path, "-> khong phai checkpoint train, khoa:", keys)
        return None
    global RESUME_ITER
    RESUME_ITER = int(obj["iter"])
    print("  [dung duoc]", path, "| iter =", RESUME_ITER)
    return Path(path)


RESUME_CKPT = None
RESUME_ITER = 0          # iter cua checkpoint tim duoc -> suy ra so epoch da xong
if ON_KAGGLE:
    print("tim checkpoint de train tiep trong /kaggle/input ...")

    # a) thu muc da bi Kaggle giai nen san. Tim theo data.pkl chu KHONG theo do
    #    sau co dinh: duong dan Kaggle Model la
    #    /kaggle/input/models/<user>/<model>/<framework>/<variation>/<version>/<ten>
    #    tuc 7 cap, dem thieu mot cap la truot.
    for f in Path("/kaggle/input").rglob("data.pkl"):
        cand = _rezip_extracted_dir(f.parent)
        if cand is not None:
            RESUME_CKPT = _usable_checkpoint(cand)
            if RESUME_CKPT is not None:
                break

    # b) tim theo ten file, tu cu the den chung chung; dung ngay khi nap duoc
    if RESUME_CKPT is None:
        for pattern in ("vietocr_checkpoint.pth", "vietocr_checkpoint.zip",
                        "vietocr_checkpoint", "*checkpoint*.pth", "*checkpoint*.zip",
                        "*.pth", "*.pt"):
            for c in sorted(Path("/kaggle/input").rglob(pattern)):
                if not c.is_file():
                    continue
                RESUME_CKPT = _usable_checkpoint(c)
                if RESUME_CKPT is not None:
                    break
            if RESUME_CKPT is not None:
                break

    if RESUME_CKPT is None:
        print("  khong thay checkpoint nao dung duoc -> train tu dau")

print("DATA_DIR   :", DATA_DIR)
print("WORK       :", WORK)
print("LMDB       :", LMDB_DIR)
print("resume tu  :", RESUME_CKPT or "(khong co - train tu dau)")


## 4. Kiểm tra dữ liệu và dựng vocab

Vocab mặc định của VietOCR đủ chữ tiếng Việt có dấu nhưng thiếu dấu câu của bộ này (`«` `»` `“` `”` `–` `…`). Nối thêm **vào cuối**, không sắp xếp lại — để chỉ số cũ giữ nguyên ý nghĩa và chỉ các hàng embedding mới là khởi tạo lại.


In [ ]:
# ============================================================================
# KIEM TRA DU LIEU + DUNG VOCAB
# ============================================================================
import unicodedata, random, os
import numpy as np, pandas as pd
from IPython.display import display

def read_labels(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n\r")
            if line:
                rel, text = line.split("\t", 1)
                rows.append((os.path.normpath(rel), unicodedata.normalize("NFC", text)))
    return rows

train_rows, val_rows, test_rows = map(read_labels, [TRAIN_FILE, VAL_FILE, TEST_FILE])
lengths = np.array([len(t) for _, t in train_rows])
MAX_LABEL_LEN = int(lengths.max())
display(pd.DataFrame({"split": ["train", "val", "test"],
                      "so_mau": [len(train_rows), len(val_rows), len(test_rows)]}))
print("Do dai nhan p50/p90/p99/max:",
      np.percentile(lengths, [50, 90, 99]).round(1).tolist(), MAX_LABEL_LEN)

# ---------------------------------------------------------------- VOCAB
# Vocab mac dinh cua VietOCR du chu tieng Viet nhung THIEU dau cau cua bo nay
# (« » “ ” – …). Phai NOI VAO CUOI: chi so 0..N-1 giu nguyen y nghia, chi cac
# hang embedding moi la khoi tao lai. Sap xep lai = moi trong so thanh vo nghia.
from vietocr.tool.config import Cfg

base_cfg = Cfg.load_config_from_name(MODEL_NAME)
DEFAULT_VOCAB = base_cfg["vocab"]

data_chars = set("".join(t for _, t in train_rows + val_rows + test_rows))
dict_chars = {unicodedata.normalize("NFC", l.rstrip("\n\r"))
              for l in open(DICT_FILE, encoding="utf-8") if l.rstrip("\n\r")}
need = (data_chars | dict_chars) - {"\n", "\r"}

extra = [c for c in sorted(need) if c not in DEFAULT_VOCAB]
VOCAB = DEFAULT_VOCAB + "".join(extra)

print(f"\nvocab mac dinh : {len(DEFAULT_VOCAB)} ky tu")
print(f"noi them       : {len(extra)} ky tu  ->  {''.join(extra)!r}")
print(f"vocab dung     : {len(VOCAB)} ky tu")
assert len(set(VOCAB)) == len(VOCAB), "vocab co ky tu trung"
assert not (need - set(VOCAB)), f"van thieu: {sorted(need - set(VOCAB))}"

# nhan hong: gt chua ky tu ngoai vi_dict.txt (vd 'ü') — dem de bao cao, khong sua
noisy = [(r, t) for r, t in train_rows if (set(t) - {" "}) - dict_chars]
print(f"\nDong train co ky tu ngoai vi_dict.txt (nhan nghi hong): {len(noisy)}")
if noisy:
    print("  vi du:", noisy[0][1][:60])

# --------------------------------------------------------- kiem tra anh
rng = random.Random(SEED)
for rel, _ in rng.sample(train_rows + val_rows + test_rows, 300):
    assert (DATA_DIR / rel).exists(), f"thieu anh {rel}"
print("Anh: OK (mau 300)")

# --------------------------------------- val nho cho eval trong luc train
VAL_SMALL = WORK / "rec_val_small.txt"
vlines = [l for l in open(VAL_FILE, encoding="utf-8").read().splitlines() if l.strip()]
VAL_SMALL.write_text("\n".join(vlines[:1000]) + "\n", encoding="utf-8")
print(f"val nho: 1000 dong -> {VAL_SMALL.name}")

# --------------------------------------------------- ty le khung -> width
from PIL import Image
ars = []
for rel, _ in rng.sample(test_rows, 800):
    w, h = Image.open(DATA_DIR / rel).size
    ars.append(w / h)
ars = np.array(ars)
print("\nAspect w/h  p50/p90/p99:", np.percentile(ars, [50, 90, 99]).round(1).tolist())
print(f"O chieu cao {IMG_HEIGHT}px:")
for w in [512, 768, 1024, 1280, 1536]:
    keep = (ars <= w / IMG_HEIGHT).mean() * 100
    mark = "  <- dang dung" if w == IMG_MAX_WIDTH else ""
    print(f"  max_width {w:5d}: giu nguyen ty le cho {keep:5.1f}% anh{mark}")


## 5. Hàm tiện ích

`evaluate()` giống hệt bản PaddleOCR (NFC, `ignore_space`, rapidfuzz) để hai bên so được với nhau.


In [ ]:
# ============================================================================
# HAM TIEN ICH: chay tien trinh co deadline, doc log TRUC TIEP, cham diem
# ============================================================================
import subprocess, json, os, re, sys, time, threading, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from rapidfuzz.distance import Levenshtein

ENV = os.environ.copy()
ENV["PYTHONUNBUFFERED"] = "1"

# Script train tu them tien to [t=<giay>] vao moi dong, con lai la log goc
# cua VietOCR:
#   iter: 000200 - train loss: 1.234 - lr: 1.00e-04 - load time: .. - gpu time: ..
#   iter: 004000 - valid loss: 0.987 - acc full seq: 0.6543 - acc per char: 0.9876
_T_RE     = re.compile(r"\[t=([\d.]+)\]")
_ITER_RE  = re.compile(r"iter: (\d+)")
_EPOCH_RE = re.compile(r"EPOCH (\d+)/(\d+) xong sau ([\d.]+)s")
_F        = r"(-?\d+\.?\d*(?:[eE][-+]?\d+)?)"
_TRAIN_RE = re.compile(rf"train loss: {_F} - lr: {_F}")
_VALID_RE = re.compile(rf"valid loss: {_F} - acc full seq: {_F} - acc per char: {_F}")
_GPU_RE   = re.compile(rf"gpu time: {_F}")


def parse_train_log(log_path):
    """-> (train_df, eval_df, epoch_df) doc tu log cua vietocr_train.py."""
    log_path = Path(log_path)
    if not log_path.exists():
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    trs, evs, eps = [], [], []
    for line in log_path.read_text(errors="ignore").splitlines():
        mt = _T_RE.search(line)
        t = float(mt.group(1)) if mt else None
        mi = _ITER_RE.search(line)
        it = int(mi.group(1)) if mi else None

        m = _TRAIN_RE.search(line)
        if m and it is not None:
            g = _GPU_RE.search(line)
            trs.append({"iter": it, "loss": float(m.group(1)), "lr": float(m.group(2)),
                        "gpu_time": float(g.group(1)) if g else None, "elapsed_s": t})
            continue
        m = _VALID_RE.search(line)
        if m and it is not None:
            evs.append({"iter": it, "valid_loss": float(m.group(1)),
                        "acc_full_seq": float(m.group(2)),
                        "acc_per_char": float(m.group(3)), "elapsed_s": t})
            continue
        m = _EPOCH_RE.search(line)
        if m:
            eps.append({"epoch": int(m.group(1)), "tong_epoch": int(m.group(2)),
                        "giay": float(m.group(3)), "elapsed_s": t})
    return pd.DataFrame(trs), pd.DataFrame(evs), pd.DataFrame(eps)


class LogTail(threading.Thread):
    """Doc log dang duoc ghi va in ra notebook, khoi phai doi het gio moi thay."""
    STEP_EVERY = 120

    def __init__(self, log_path, total_epochs, t_start):
        super().__init__(daemon=True)
        self.path, self.total, self.t_start = Path(log_path), total_epochs, t_start
        self.stop_evt, self.pos, self.last_print = threading.Event(), 0, 0.0
        self.n_eval = 0

    def _emit(self, line):
        now = time.time()
        m = _VALID_RE.search(line)
        if m:
            self.n_eval += 1
            it = _ITER_RE.search(line)
            print(f"  [EVAL {self.n_eval}] iter {it.group(1) if it else '?'}"
                  f"  val_acc {float(m.group(2))*100:6.2f}%"
                  f"  acc/char {float(m.group(3))*100:6.2f}%"
                  f"  loss {float(m.group(1)):.3f}"
                  f"  | da train {hms(now - self.t_start)}"
                  f"  | ngan sach con {hms(time_left())}", flush=True)
            return
        m = _EPOCH_RE.search(line)
        if m:
            print(f"  >>> XONG EPOCH {m.group(1)}/{m.group(2)} sau {hms(float(m.group(3)))}"
                  f"  | ngan sach con {hms(time_left())}", flush=True)
            return
        m = _TRAIN_RE.search(line)
        if m:
            if now - self.last_print < self.STEP_EVERY:
                return
            self.last_print = now
            it = _ITER_RE.search(line)
            print(f"    iter {it.group(1) if it else '?'}"
                  f"  loss {float(m.group(1)):7.4f}"
                  f"  lr {float(m.group(2)):.2e}"
                  f"  | {hms(now - self.t_start)}", flush=True)
            return
        if "Traceback" in line or "Error" in line or "out of memory" in line.lower():
            print("    !", line.strip()[-160:], flush=True)

    def _drain(self):
        if not self.path.exists():
            return
        with open(self.path, "r", encoding="utf-8", errors="ignore") as f:
            f.seek(self.pos)
            chunk = f.read()
            self.pos = f.tell()
        for line in chunk.splitlines():
            self._emit(line)

    def run(self):
        while not self.stop_evt.wait(5):
            self._drain()
        self._drain()


def run_process(args, log_name, timeout=None, watch_epochs=None, cwd=None):
    """-> (giay, hoan_thanh). Het gio -> terminate, checkpoint van con."""
    log_path = LOG_DIR / log_name
    start, tail = time.time(), None
    with open(log_path, "w", encoding="utf-8") as log:
        p = subprocess.Popen([str(x) for x in args], cwd=str(cwd or SCRATCH),
                             stdout=log, stderr=subprocess.STDOUT, text=True, env=ENV)
        if watch_epochs:
            tail = LogTail(log_path, watch_epochs, start); tail.start()
        try:
            rc = p.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            p.terminate()
            try:
                p.wait(120)
            except subprocess.TimeoutExpired:
                p.kill()
            if tail:
                tail.stop_evt.set(); tail.join(15)
            print(f"  [het ngan sach] dung {log_name} sau {hms(time.time()-start)}")
            return time.time() - start, False
        finally:
            if tail and tail.is_alive():
                tail.stop_evt.set(); tail.join(15)
    if rc != 0:
        print("\n".join(log_path.read_text(errors="ignore").splitlines()[-60:]))
        raise RuntimeError("Loi: " + " ".join(map(str, args)))
    return time.time() - start, True


# ------------------------------------------------------------- cham diem
def read_predictions(path):
    preds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                parts = parts + [""]
            preds[os.path.normpath(parts[0])] = unicodedata.normalize("NFC", parts[1])
    return preds


def metric_text(s):
    s = unicodedata.normalize("NFC", s)
    return s.replace(" ", "") if IGNORE_SPACE else s


def evaluate(label_file, pred_txt, jsonl_path=None):
    """Y HET ham cua notebook PaddleOCR -> hai ben so duoc voi nhau."""
    gt_rows, preds = read_labels(label_file), read_predictions(pred_txt)
    recs, dists, correct = [], [], 0
    for rel, gt in gt_rows:
        pred = preds.get(rel, "")
        a, b = metric_text(pred), metric_text(gt)
        correct += int(a == b)
        dists.append(Levenshtein.normalized_distance(a, b))
        recs.append({"image": rel.replace(os.sep, "/"), "gt": gt, "pred": pred})
    res = {"acc": correct / len(gt_rows),
           "norm_edit_dis": 1 - float(np.mean(dists)),
           "n": len(gt_rows),
           "missing_pred": sum(1 for r, _ in gt_rows if r not in preds)}
    if jsonl_path:
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for r in recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return res, recs


print("helpers OK")


## 6. Sinh config và script

Train/predict chạy ở tiến trình riêng: notebook không giữ bộ nhớ GPU, và áp được deadline cứng.


In [ ]:
# ============================================================================
# SINH CONFIG + 2 SCRIPT CHAY QUA SUBPROCESS
#   Chay train/predict o tien trinh rieng de notebook khong giu bo nho GPU,
#   va de ap deadline cung duoc (run_process tu terminate truoc khi Kaggle cat).
# ============================================================================
import yaml, json, math

N_TRAIN = len(train_rows)
ITERS_PER_EPOCH = max(1, N_TRAIN // BATCH_SIZE)
TOTAL_ITERS = ITERS_PER_EPOCH * TARGET_EPOCHS
VALID_EVERY = max(1, ITERS_PER_EPOCH // max(1, EVAL_PER_EPOCH))

FT_WEIGHTS  = OUTPUT_DIR / "vietocr_finetune.pth"
CKPT_PATH   = OUTPUT_DIR / "vietocr_checkpoint.pth"
EPOCH_DIR   = OUTPUT_DIR / "epochs"; EPOCH_DIR.mkdir(parents=True, exist_ok=True)


def ckpt_epoch(path):
    """-> (iter, so epoch da xong) doc tu checkpoint. Khong co file thi (0, 0)."""
    if path is None or not Path(path).exists():
        return 0, 0
    import torch
    it = int(torch.load(path, map_location="cpu", weights_only=False).get("iter", 0))
    return it, it // ITERS_PER_EPOCH


# Epoch da train xong o cac phien TRUOC. Phai biet con so nay TRUOC khi hieu chuan,
# vi FIT_EPOCHS la moc epoch TUYET DOI (script train chay range(done+1, epochs+1)),
# khong phai "so epoch chay them trong phien nay".
if CKPT_PATH.exists():                       # chay lai trong cung phien
    RESUME_ITER, DONE_EPOCHS = ckpt_epoch(CKPT_PATH)
else:                                        # lay tu checkpoint phien truoc (cell 3)
    DONE_EPOCHS = RESUME_ITER // ITERS_PER_EPOCH

if DONE_EPOCHS:
    print(f"checkpoint dang o iter {RESUME_ITER} -> da xong {DONE_EPOCHS}/{TARGET_EPOCHS} epoch"
          f"  |  phien nay train tiep tu epoch {DONE_EPOCHS + 1}")
else:
    print("khong co checkpoint -> train tu epoch 1")


def make_config(vocab, weights=None, tag="finetune", pretrained_geometry=False):
    """pretrained_geometry=True: giu NGUYEN hinh hoc goc cua trong so pretrained.

    Bat buoc cho baseline. cnn.ss/ks la stride cua lop pooling — pooling KHONG co
    tham so hoc duoc nen khong nam trong state_dict: doi ss/ks van load_state_dict
    tron tru, khong mot canh bao nao, chi la dac trung CNN tinh sai hoan toan ->
    acc ~ 0. Cung ly do voi image_height: trong so pretrained hoc o h=32, cho an
    h=64 la lech phan bo dau vao.
    """
    base = Cfg.load_config_from_name(MODEL_NAME)
    cfg = dict(base)
    cfg["vocab"] = vocab
    cfg["device"] = "cuda:0"
    cfg["cnn"] = dict(cfg["cnn"])
    if pretrained_geometry:
        img_h = base["dataset"]["image_height"]
        img_min = base["dataset"]["image_min_width"]
        img_max = base["dataset"]["image_max_width"]
    else:
        cfg["cnn"]["ss"] = CNN_SS; cfg["cnn"]["ks"] = CNN_KS
        img_h, img_min, img_max = IMG_HEIGHT, IMG_MIN_WIDTH, IMG_MAX_WIDTH
    # duong dan nhan de TUYET DOI: OCRDataset lam os.path.join(root_dir, annotation),
    # ma join voi duong dan tuyet doi thi tra ve chinh no -> tranh duoc viec
    # /kaggle/input chi doc nen khong dat file val nho vao trong do duoc.
    cfg["dataset"] = {
        "name": "vi", "data_root": str(DATA_DIR),
        "train_annotation": str(TRAIN_FILE), "valid_annotation": str(VAL_SMALL),
        "image_height": img_h, "image_min_width": img_min,
        "image_max_width": img_max,
    }
    cfg["dataloader"] = {"num_workers": 2, "pin_memory": True}
    cfg["trainer"] = {
        "batch_size": BATCH_SIZE, "print_every": 100, "valid_every": VALID_EVERY,
        "iters": TOTAL_ITERS, "export": str(FT_WEIGHTS), "checkpoint": str(CKPT_PATH),
        "log": str(LOG_DIR / "vietocr_inner.log"), "metrics": 1000,
    }
    cfg["optimizer"] = {"max_lr": MAX_LR, "pct_start": PCT_START}
    cfg["aug"] = {"image_aug": True, "masked_language_model": True}
    cfg["predictor"] = {"beamsearch": False}
    if weights is not None:
        cfg["weights"] = str(weights)
    path = CONFIG_DIR / f"{tag}.yml"
    yaml.safe_dump(cfg, open(path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
    return path


# Baseline PHAI dung vocab mac dinh (Predictor nap bang load_state_dict khong co
# strict=False, doi vocab la lech shape lop embedding -> no) VA phai giu nguyen
# hinh hoc goc h=32 / ss cuoi [1,1]. Dua trong so pretrained vao hinh hoc h=64 +
# ss cuoi [2,1] thi no van nap duoc nhung acc tut ve ~0 — do la con so vo nghia,
# khong phai "baseline yeu".
CFG_BASELINE = make_config(DEFAULT_VOCAB, weights=None, tag="baseline",
                           pretrained_geometry=True)
CFG_FINETUNE = make_config(VOCAB, weights=FT_WEIGHTS, tag="finetune")

# Config rieng CHI de hieu chuan toc do suy luan, chay TRUOC khi train:
#   - hinh hoc = hinh hoc that (h=64, ss cuoi [2,1]) -> do dung tot do se chay
#   - vocab mac dinh + weights = URL pretrained -> co file that de nap
# CFG_FINETUNE khong dung duoc o day vi weights cua no tro vao vietocr_finetune.pth,
# file chi ton tai SAU khi train xong. Do bang chu voi ket qua rac cung khong sao:
# ta chi lay giay/anh, khong lay do chinh xac.
CFG_SPEED = make_config(DEFAULT_VOCAB, weights=None, tag="speed")

_c = yaml.safe_load(open(CFG_FINETUNE, encoding="utf-8"))
seq_len = (IMG_MAX_WIDTH // 4) * (IMG_HEIGHT // 32)
print(f"iter/epoch          : {ITERS_PER_EPOCH}  ({N_TRAIN} dong / batch {BATCH_SIZE})")
print(f"tong iter {TARGET_EPOCHS} epoch: {TOTAL_ITERS}")
print(f"valid_every         : {VALID_EVERY}  ({EVAL_PER_EPOCH} lan do / epoch)")
print(f"vocab               : {len(_c['vocab'])} ky tu")
print(f"cnn.ss              : {_c['cnn']['ss']}")
print(f"  tich stride cao   : {math.prod(s[0] for s in CNN_SS)}  -> feature height "
      f"{IMG_HEIGHT // math.prod(s[0] for s in CNN_SS)}")
print(f"  tich stride rong  : {math.prod(s[1] for s in CNN_SS)}")
print(f"seq len encoder toi da: {seq_len}  (o width {IMG_MAX_WIDTH})")
assert IMG_HEIGHT % math.prod(s[0] for s in CNN_SS) == 0, "IMG_HEIGHT khong chia het cho tich stride"
assert IMG_HEIGHT // math.prod(s[0] for s in CNN_SS) == 2, (
    "feature height phai = 2 de chuoi encoder khong phinh ra; sua CNN_SS")
assert MAX_LABEL_LEN < _c.get("transformer", {}).get("max_seq_length", 1024) - 2, \
    f"max_seq_length qua nho so voi nhan dai nhat {MAX_LABEL_LEN}"

# ============================================================================
# script 1: train — chay theo tung epoch de co checkpoint moi epoch (muc 2.4)
#           va de kiem tra deadline sau moi epoch.
# ============================================================================
(SCRIPT_DIR / "vietocr_train.py").write_text(NP2_SHIM + '''
import argparse, os, sys, time, yaml, torch
T0 = time.time()

class _Stamp:
    """Them [t=<giay>] vao dau moi dong -> parse duoc thoi gian tu log."""
    def __init__(self, s): self.s, self.nl = s, True
    def write(self, x):
        for part in x.splitlines(True):
            if self.nl and part.strip():
                self.s.write("[t=%.1f] " % (time.time() - T0))
            self.s.write(part)
            self.nl = part.endswith("\\n")
    def flush(self): self.s.flush()
sys.stdout = _Stamp(sys.stdout)
sys.stderr = sys.stdout

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--epochs", type=int, required=True)
ap.add_argument("--iters-per-epoch", type=int, required=True)
ap.add_argument("--epoch-dir", required=True)
ap.add_argument("--resume", default=None)
ap.add_argument("--deadline", type=float, default=1e9, help="giay ke tu khi script bat dau")
ap.add_argument("--seed", type=int, default=2026)
a = ap.parse_args()

torch.manual_seed(a.seed)
from vietocr.tool.config import Cfg
from vietocr.model.trainer import Trainer

cfg = Cfg(yaml.safe_load(open(a.config, encoding="utf-8")))
print("dang dung LMDB trong:", os.getcwd())

def load_checkpoint_compat(trainer, filename):
    """Thay cho Trainer.load_checkpoint — ham do cua vietocr 0.3.13 bi loi.

    Trainer.__init__ dung AdamW + OneCycleLR, phan ScheduledOptim da bi comment.
    Nhung load_checkpoint van con sot dong code cu:
        optim = ScheduledOptim(Adam(...), d_model, **self.config["optimizer"])
    config["optimizer"] bay gio la {max_lr, pct_start} cua OneCycleLR, khong khop
    chu ky ScheduledOptim(optimizer, d_model, init_lr, n_warmup_steps):
        TypeError: ScheduledOptim.__init__() got an unexpected keyword argument "max_lr"
    Bien `optim` do KHONG duoc dung o bat ky dong nao phia sau — no la code chet.
    Ham nay lam dung phan con lai, khong dung ScheduledOptim.
    """
    ck = torch.load(filename, map_location=torch.device(trainer.device),
                    weights_only=False)
    missing = [k for k in ("state_dict", "optimizer", "iter") if k not in ck]
    if missing:
        raise KeyError("checkpoint thieu khoa %s (co: %s)" % (missing, list(ck)[:8]))
    trainer.model.load_state_dict(ck["state_dict"])
    trainer.optimizer.load_state_dict(ck["optimizer"])
    trainer.iter = ck["iter"]
    trainer.train_losses = ck.get("train_losses", [])


resume = a.resume and os.path.exists(a.resume)
trainer = Trainer(cfg, pretrained=not resume)
if resume:
    load_checkpoint_compat(trainer, a.resume)
    print("resume tu %s, iter hien tai %d" % (a.resume, trainer.iter))
    print("LUU Y: lich LR OneCycle khoi dong lai sau khi resume (VietOCR khong "
          "luu trang thai scheduler trong checkpoint).")

done = trainer.iter // a.iters_per_epoch
print("checkpoint o iter %d -> da xong %d epoch" % (trainer.iter, done))
print("phien nay train epoch %d -> %d" % (done + 1, a.epochs))
if done >= a.epochs:
    print("KHONG CO GI DE TRAIN: da xong %d epoch ma --epochs chi la %d. "
          "--epochs phai la MOC TUYET DOI, khong phai so epoch chay them."
          % (done, a.epochs))
trainer.num_iters = a.iters_per_epoch
last_epoch_s = 0.0

for ep in range(done + 1, a.epochs + 1):
    t = time.time()
    left = a.deadline - (time.time() - T0)
    if ep > done + 1 and left < last_epoch_s * 1.05:
        print("DUNG SOM: con %.0fs, mot epoch can ~%.0fs" % (left, last_epoch_s))
        break
    trainer.train()
    last_epoch_s = time.time() - t
    w = os.path.join(a.epoch_dir, "epoch_%d.pth" % ep)
    trainer.save_weights(w)
    trainer.save_checkpoint(cfg["trainer"]["checkpoint"])
    print("=== EPOCH %d/%d xong sau %.1fs -> %s" % (ep, a.epochs, last_epoch_s, w))
    print("    (iter hien tai %d)" % trainer.iter)

trainer.save_checkpoint(cfg["trainer"]["checkpoint"])
print("KET THUC: iter=%d, tong %.1fs" % (trainer.iter, time.time() - T0))
''', encoding="utf-8")

# ============================================================================
# script 2: predict
# ============================================================================
(SCRIPT_DIR / "vietocr_predict.py").write_text(NP2_SHIM + '''
import argparse, os, sys, time, yaml, torch
from PIL import Image

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--weights", default=None)
ap.add_argument("--list", required=True)
ap.add_argument("--data-root", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--limit", type=int, default=0)
ap.add_argument("--batch", type=int, default=24)
ap.add_argument("--beamsearch", action="store_true")
a = ap.parse_args()

from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

cfg = Cfg(yaml.safe_load(open(a.config, encoding="utf-8")))
if a.weights:
    cfg["weights"] = a.weights
cfg["predictor"]["beamsearch"] = a.beamsearch
pred = Predictor(cfg)

rels = []
for line in open(a.list, encoding="utf-8"):
    line = line.rstrip("\\n\\r")
    if line:
        rels.append(line.split("\\t", 1)[0])
if a.limit:
    rels = rels[:a.limit]

t0 = time.time()
with open(a.out, "w", encoding="utf-8") as f:
    for i in range(0, len(rels), a.batch):
        chunk = rels[i:i + a.batch]
        imgs = [Image.open(os.path.join(a.data_root, r)).convert("RGB") for r in chunk]
        if a.beamsearch:
            texts = [pred.predict(im) for im in imgs]
        else:
            texts = pred.predict_batch(imgs)
        for r, t in zip(chunk, texts):
            f.write("%s\\t%s\\n" % (r, t.replace("\\t", " ").replace("\\n", " ")))
        if (i // a.batch) % 20 == 0:
            n = i + len(chunk)
            print("  %d/%d anh  %.3f s/anh" % (n, len(rels), (time.time() - t0) / n), flush=True)
print("XONG %d anh trong %.1fs (%.4f s/anh)" % (len(rels), time.time() - t0,
                                                (time.time() - t0) / max(1, len(rels))))
''', encoding="utf-8")

print("\nconfig + script da san sang:", CFG_BASELINE.name, CFG_FINETUNE.name)


## 7. Hiệu chuẩn thời gian

Đo tốc độ thật rồi tính xem bao nhiêu epoch vừa ngân sách. Đây là cell quyết định `FIT_EPOCHS`.


In [ ]:
# ============================================================================
# HIEU CHUAN THOI GIAN — do that, khong doan
#   Muc tieu: notebook luon chay HET moi cell trong 1 phien. So epoch bi co lai
#   cho vua ngan sach, chu khong de Kaggle cat ngang o gio thu 12.
# ============================================================================
import shutil, yaml

CALIB_DIR = SCRATCH / "calib"; CALIB_DIR.mkdir(parents=True, exist_ok=True)

# Chan truoc: Predictor nap cfg["weights"] cung, khong co strict=False. Tro vao
# file chua ton tai la chet giua chung sau khi da nap xong model.
_w = yaml.safe_load(open(CFG_SPEED, encoding="utf-8"))["weights"]
assert str(_w).startswith("http") or Path(_w).exists(), \
    f"CFG_SPEED tro weights vao file chua ton tai: {_w}"

# --- 1. do toc do SUY LUAN (decode tu hoi quy, phai do chu khong uoc luong) ---
calib_pred = CALIB_DIR / "calib_pred.txt"
sec_infer, _ = run_process(
    [sys.executable, SCRIPT_DIR / "vietocr_predict.py",
     # Do tren CFG_SPEED: cung hinh hoc voi luc chay that (h=64/w<=1024) nhung
     # weights la URL pretrained nen nap duoc TRUOC khi train. Do tren CFG_BASELINE
     # thi sai vi baseline chay h=32/w<=512 — nhanh gan gap doi thuc te, SEC_PER_IMG
     # bi uoc thap, ngan sach tinh sai va Kaggle cat ngang phien.
     "--config", CFG_SPEED, "--list", TEST_FILE, "--data-root", DATA_DIR,
     "--out", calib_pred, "--limit", CALIB_IMAGES],
    "calib_predict.log")
n_test = len(test_rows)
# Lay tu dong "XONG ... s/anh" cua script — dong ho cua no bat dau SAU khi nap
# model, nen khong bi chi phi khoi dong ~20-30s lam sai lech uoc luong.
_m = re.search(r"XONG \d+ anh trong [\d.]+s \(([\d.]+) s/anh\)",
               (LOG_DIR / "calib_predict.log").read_text(errors="ignore"))
SEC_PER_IMG = float(_m.group(1)) if _m else sec_infer / CALIB_IMAGES
MODEL_LOAD_S = max(0.0, sec_infer - SEC_PER_IMG * CALIB_IMAGES)
print(f"suy luan : {SEC_PER_IMG:.4f} s/anh  (nap model rieng: {MODEL_LOAD_S:.0f}s)")

# --- 2. do toc do TRAIN ---
_cc = yaml.safe_load(open(CFG_FINETUNE, encoding="utf-8"))
_cc["trainer"]["export"] = str(CALIB_DIR / "calib.pth")
_cc["trainer"]["checkpoint"] = str(CALIB_DIR / "calib_ckpt.pth")
_cc["trainer"]["valid_every"] = 10 ** 9        # khong eval trong luc hieu chuan
_cc["trainer"]["iters"] = CALIB_ITERS
_cc["trainer"]["print_every"] = 25             # nhieu diem hon -> do doc chinh xac hon
CFG_CALIB = CONFIG_DIR / "calib.yml"
yaml.safe_dump(_cc, open(CFG_CALIB, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

sec_calib, _ = run_process(
    [sys.executable, SCRIPT_DIR / "vietocr_train.py",
     "--config", CFG_CALIB, "--epochs", 1, "--iters-per-epoch", CALIB_ITERS,
     "--epoch-dir", CALIB_DIR, "--seed", SEED],
    "calib_train.log", cwd=LMDB_DIR)

_tr, _, _ = parse_train_log(LOG_DIR / "calib_train.log")
# Do doc giua diem dau va diem cuoi, bo 25 iter dau (nap model, dung LMDB,
# warm-up cuDNN) — neu tinh ca phan do se doi thanh tram giay moi epoch.
_w = _tr[_tr["iter"] > 25] if len(_tr) else _tr
if len(_w) >= 2:
    SEC_PER_ITER = float((_w["elapsed_s"].iloc[-1] - _w["elapsed_s"].iloc[0]) /
                         (_w["iter"].iloc[-1] - _w["iter"].iloc[0]))
else:
    SEC_PER_ITER = sec_calib / CALIB_ITERS
    print("  (khong doc duoc du diem trong log, dung thoi gian tong — se hoi cao)")
SEC_PER_EPOCH = SEC_PER_ITER * ITERS_PER_EPOCH
SEC_PER_EVAL = 1000 * SEC_PER_IMG * 0.5      # val nho 1000 dong, batch nen nhanh hon

# --- 3. dung bang du toan ---
fixed = {
    f"baseline suy luan {n_test} anh": n_test * SEC_PER_IMG + MODEL_LOAD_S,
    f"finetune suy luan {n_test} anh": n_test * SEC_PER_IMG + MODEL_LOAD_S,
    f"ablation suy luan 2 x {len(val_rows)} anh": 2 * (len(val_rows) * SEC_PER_IMG + MODEL_LOAD_S),
    "eval trong luc train": SEC_PER_EVAL * EVAL_PER_EPOCH * (TARGET_EPOCHS - DONE_EPOCHS),
    "phan tich + dong goi": 8 * 60,
    "du phong": RESERVE_MIN * 60,
}
FIXED_COST = sum(fixed.values())
budget = time_left()

print(f"\n{'='*62}\nDU TOAN\n{'='*62}")
print(f"iter/epoch          : {ITERS_PER_EPOCH}")
print(f"giay/iter (do that) : {SEC_PER_ITER:.3f}")
print(f"=> 1 epoch          : {hms(SEC_PER_EPOCH)}")
print(f"\nChi phi co dinh:")
for k, v in fixed.items():
    print(f"  {k:32s}: {hms(v)}")
print(f"  {'-'*32}  {'-'*8}\n  {'tong':32s}: {hms(FIXED_COST)}")

train_budget = budget - FIXED_COST
print(f"\nNgan sach con {hms(budget)} -> danh cho train: {hms(train_budget)}")

# FIT_EPOCHS la MOC EPOCH TUYET DOI se dat toi sau phien nay, KHONG phai so epoch
# chay them. Script train chay range(done+1, epochs+1) nen neu truyen so tuong doi
# thi checkpoint o epoch 7 + FIT_EPOCHS=3 -> range(8, 4) -> rong, khong train gi ca
# ma cung khong bao loi.
FIT_EPOCHS, cum = DONE_EPOCHS, 0.0
for e in range(DONE_EPOCHS + 1, TARGET_EPOCHS + 1):
    cum += SEC_PER_EPOCH
    fits = cum <= train_budget
    if fits:
        FIT_EPOCHS = e
    print(f"  epoch {e:2d} -> tich luy {hms(cum)}  [{'vua' if fits else 'KHONG VUA'}]")

if DONE_EPOCHS:
    print(f"\n=> Da co {DONE_EPOCHS} epoch. Phien nay train epoch "
          f"{DONE_EPOCHS + 1} -> {FIT_EPOCHS}  (muc tieu {TARGET_EPOCHS}).")
else:
    print(f"\n=> Train epoch 1 -> {FIT_EPOCHS}  (muc tieu {TARGET_EPOCHS}).")
if FIT_EPOCHS <= DONE_EPOCHS:
    print("   !! Khong du gio cho ca 1 epoch. Notebook se bo qua phan train va\n"
          "      chi bao cao baseline. Cach xu ly: giam BATCH_SIZE hoac IMG_MAX_WIDTH,\n"
          "      hoac doi Accelerator sang P100.")
elif FIT_EPOCHS < TARGET_EPOCHS:
    print(f"   Con {TARGET_EPOCHS - FIT_EPOCHS} epoch cho phien sau:\n"
          "     1) Save Version notebook nay\n"
          "     2) Notebook moi -> Add Data -> chon output cua lan chay nay\n"
          "     3) Run All — script tu tim vietocr_checkpoint.pth va train tiep")

TRAIN_DEADLINE = max(0.0, train_budget)
shutil.rmtree(CALIB_DIR, ignore_errors=True)


## 8. Baseline — mục 2.1


In [ ]:
# ============================================================================
# 2.1 BASELINE — VietOCR goc (vgg_transformer pretrained), chua fine-tune,
#     tren rec_test.txt. Dung vocab MAC DINH cua no, khong sua gi.
#
# Khac han baseline cua PaddleOCR: vocab cua VietOCR da co san day du chu
# tieng Viet co dau, nen acc phai cao hon 4.7% cua latin_PP-OCRv5 nhieu.
# Neu van ~0 thi la loi nap trong so hoac vocab, khong phai ket qua that.
# ============================================================================
BASELINE_TXT   = RESULTS_DIR / "baseline_test.txt"
BASELINE_JSONL = RESULTS_DIR / "pred_test_baseline.jsonl"

baseline_seconds, _ = run_process(
    [sys.executable, SCRIPT_DIR / "vietocr_predict.py",
     "--config", CFG_BASELINE, "--list", TEST_FILE, "--data-root", DATA_DIR,
     "--out", BASELINE_TXT],
    "baseline_test.log")

baseline_metric, _ = evaluate(TEST_FILE, BASELINE_TXT, BASELINE_JSONL)
baseline_metric["inference_seconds"] = round(baseline_seconds, 1)

print(f"Baseline: VietOCR {MODEL_NAME} (pretrained) | tap: rec_test.txt")
display(pd.DataFrame([baseline_metric]))
print(f"ngan sach con {hms(time_left())}")

if baseline_metric["acc"] < 0.01:
    print("\n!! acc gan bang 0 — kiem tra lai truoc khi train:\n"
          "   - vocab cua config baseline co phai DEFAULT_VOCAB khong\n"
          "   - Predictor co nap dung file trong so khong (xem logs/baseline_test.log)")


## 9. Fine-tune — mục 2.2


In [ ]:
# ============================================================================
# 2.2 FINE-TUNE — chay den FIT_EPOCHS (da tinh o cell hieu chuan), co deadline
#     cung nen khong bao gio bi Kaggle cat ngang.
#
# Trong luc train, cell nay in truc tiep:
#   - moi 2 phut : 1 dong tien do (iter, loss, lr, thoi gian)
#   - moi lan eval: val_acc + acc/char + ngan sach con lai
#   - moi epoch  : moc hoan thanh + duong dan checkpoint
# ============================================================================
TRAIN_LOG = "finetune_train.log"
HIST_DIR = OUTPUT_DIR / "history"; HIST_DIR.mkdir(parents=True, exist_ok=True)
if (LOG_DIR / TRAIN_LOG).exists():          # giu log phien truoc -> duong cong lien mach
    shutil.copy(LOG_DIR / TRAIN_LOG, HIST_DIR / f"train_{int(time.time())}.log")

resume = CKPT_PATH if CKPT_PATH.exists() else RESUME_CKPT
if resume is not None and Path(resume) != CKPT_PATH:
    shutil.copy(resume, CKPT_PATH)          # keo checkpoint phien truoc vao thu muc lam viec
    resume = CKPT_PATH
    print(f"resume tu dataset phien truoc: {resume}")
    print(f"  iter {RESUME_ITER} -> da xong {DONE_EPOCHS} epoch, train tiep tu epoch {DONE_EPOCHS + 1}")

time_file = OUTPUT_DIR / "train_seconds.txt"
train_seconds = float(time_file.read_text()) if time_file.exists() else 0.0

if FIT_EPOCHS <= DONE_EPOCHS:
    print(f"Bo qua train: da xong {DONE_EPOCHS} epoch va khong du ngan sach cho epoch "
          f"{DONE_EPOCHS + 1} (xem cell hieu chuan).")
    done_epoch = DONE_EPOCHS
else:
    print(f"[train] epoch {DONE_EPOCHS + 1} -> {FIT_EPOCHS}  (muc tieu {TARGET_EPOCHS})"
          f"  |  h={IMG_HEIGHT} w<={IMG_MAX_WIDTH}  bs={BATCH_SIZE}  lr={MAX_LR:.1e}\n"
          f"        deadline {hms(TRAIN_DEADLINE)}  |  ngan sach con {hms(time_left())}\n")
    secs, done = run_process(
        [sys.executable, SCRIPT_DIR / "vietocr_train.py",
         "--config", CFG_FINETUNE, "--epochs", FIT_EPOCHS,
         "--iters-per-epoch", ITERS_PER_EPOCH, "--epoch-dir", EPOCH_DIR,
         "--resume", CKPT_PATH, "--deadline", TRAIN_DEADLINE, "--seed", SEED],
        TRAIN_LOG, timeout=max(600, TRAIN_DEADLINE + 300),
        watch_epochs=TARGET_EPOCHS, cwd=LMDB_DIR)
    train_seconds += secs
    time_file.write_text(str(train_seconds))
    # Dem tu CHECKPOINT chu khong tu so file epoch_*.pth: phien moi co EPOCH_DIR
    # rong nen dem file se ra 0 du checkpoint da o epoch 7.
    _it, done_epoch = ckpt_epoch(CKPT_PATH)
    print(f"\nSau phien nay: iter {_it} -> {done_epoch}/{TARGET_EPOCHS} epoch, "
          f"tong {hms(train_seconds)}")
    if done_epoch < TARGET_EPOCHS:
        print("\n>>> CHUA TRAIN DU. De chay tiep:\n"
              "    1) Save Version notebook nay\n"
              "    2) Notebook moi -> Add Data -> chon output cua lan chay nay\n"
              "    3) Run All — script tu tim vietocr_checkpoint.pth va train tiep")

# VietOCR luu export_weights moi khi acc tren val tot len -> do la 'best'
FINAL_WEIGHTS = FT_WEIGHTS if FT_WEIGHTS.exists() else None
if FINAL_WEIGHTS is None:
    cands = sorted(EPOCH_DIR.glob("epoch_*.pth"), key=lambda p: int(p.stem.split("_")[1]))
    FINAL_WEIGHTS = cands[-1] if cands else None
print("trong so dung de bao cao:", FINAL_WEIGHTS.name if FINAL_WEIGHTS else "(chua co)")
print("ngan sach con", hms(time_left()))


## 10. Số liệu từng bước train


In [ ]:
# ============================================================================
# SO LIEU TUNG BUOC TRAIN -> results/training_curve.csv + .png
#   Gop log cua tat ca cac phien (history/) thanh 1 duong cong lien tuc.
# ============================================================================
import matplotlib.pyplot as plt

def collect_history(log_dir, hist_dir, cur_log):
    logs = sorted(Path(hist_dir).glob("train_*.log"), key=lambda p: p.stat().st_mtime)
    logs.append(Path(log_dir) / cur_log)
    trs, evs, eps, off = [], [], [], 0.0
    for lg in logs:
        t, e, p = parse_train_log(lg)
        if not len(t) and not len(e):
            continue
        for df in (t, e, p):
            if len(df) and "elapsed_s" in df:
                df["elapsed_s"] = df["elapsed_s"].astype(float) + off
        off = max([df["elapsed_s"].max() for df in (t, e, p) if len(df)] or [off])
        trs.append(t); evs.append(e); eps.append(p)
    cat = lambda xs: (pd.concat([x for x in xs if len(x)], ignore_index=True)
                      if any(len(x) for x in xs) else pd.DataFrame())
    tr, ev, ep = cat(trs), cat(evs), cat(eps)
    for df, key in [(tr, "iter"), (ev, "iter"), (ep, "epoch")]:
        if len(df):
            df.drop_duplicates(key, keep="last", inplace=True)
            df.sort_values(key, inplace=True)
            df.reset_index(drop=True, inplace=True)
    if len(ev):
        ev["phut"] = (ev["elapsed_s"] / 60).round(1)
        ev["val_acc_%"] = (ev["acc_full_seq"] * 100).round(2)
        ev["epoch"] = np.ceil(ev["iter"] / ITERS_PER_EPOCH).astype(int)
    return tr, ev, ep

train_hist, eval_hist, epoch_hist = collect_history(LOG_DIR, HIST_DIR, TRAIN_LOG)

if len(eval_hist) == 0:
    print("Chua co diem eval nao trong log — train them roi chay lai cell nay.")
else:
    show = eval_hist[["epoch", "iter", "val_acc_%", "acc_per_char", "valid_loss", "phut"]].copy()
    show.columns = ["epoch", "iter", "val_acc (%)", "acc/char", "valid_loss", "phut_da_train"]
    print(f"{len(show)} diem do tren tap val ({EVAL_PER_EPOCH} lan/epoch)")
    display(show.reset_index(drop=True))

    if len(epoch_hist):
        epoch_hist["gio_tich_luy"] = (epoch_hist["giay"].cumsum() / 3600).round(2)
        print("\nThoi gian tung epoch:")
        display(epoch_hist[["epoch", "giay", "gio_tich_luy"]])
        epoch_hist.to_csv(RESULTS_DIR / "epoch_time.csv", index=False)

    if len(train_hist):
        per_ep = (train_hist.assign(epoch=np.ceil(train_hist["iter"] / ITERS_PER_EPOCH).astype(int))
                  .groupby("epoch").agg(loss_tb=("loss", "mean"), lr=("lr", "last"),
                                        so_diem=("iter", "count")).round(4))
        per_ep["val_acc_%"] = eval_hist.groupby("epoch")["val_acc_%"].max()
        per_ep["acc_per_char"] = eval_hist.groupby("epoch")["acc_per_char"].max().round(4)
        if len(epoch_hist):
            per_ep["giay"] = epoch_hist.set_index("epoch")["giay"]
        print("\nTong hop theo epoch:")
        display(per_ep)
        per_ep.to_csv(RESULTS_DIR / "per_epoch.csv")
        train_hist.to_csv(RESULTS_DIR / "train_steps.csv", index=False)
    eval_hist.to_csv(RESULTS_DIR / "training_curve.csv", index=False)

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    if len(train_hist):
        ax[0].plot(train_hist["iter"], train_hist["loss"], lw=.6, alpha=.35, color="C0")
        ax[0].plot(train_hist["iter"], train_hist["loss"].rolling(25, min_periods=1).mean(),
                   lw=1.8, color="C0", label="loss (TB truot 25)")
        ax[0].legend()
    ax[0].set(xlabel="iter", ylabel="train loss", title="Loss khi train")

    ax[1].plot(eval_hist["iter"], eval_hist["val_acc_%"], "o-", color="C2")
    ax[1].set(xlabel="iter", ylabel="val accuracy (%)",
              title=f"Accuracy tren val (dinh {eval_hist['val_acc_%'].max():.2f}%)")
    ax[1].grid(alpha=.3)

    ax[2].plot(eval_hist["phut"], eval_hist["val_acc_%"], "o-", color="C3", label="acc (%)")
    ax[2].plot(eval_hist["phut"], eval_hist["acc_per_char"] * 100, "s--", color="C4",
               label="acc/char x100")
    ax[2].set(xlabel="phut da train", ylabel="%", title="Accuracy theo thoi gian")
    ax[2].legend(); ax[2].grid(alpha=.3)

    bounds = eval_hist.drop_duplicates("epoch", keep="first")
    for a_, col in zip(ax, ["iter", "iter", "phut"]):
        for x in bounds[col]:
            a_.axvline(x, color="gray", lw=.5, ls=":")
    fig.suptitle(f"VietOCR {MODEL_NAME} — h={IMG_HEIGHT}, w<={IMG_MAX_WIDTH}, "
                 f"bs={BATCH_SIZE}, lr={MAX_LR:.1e}", y=1.02)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "training_curve.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("da luu: training_curve.csv / train_steps.csv / per_epoch.csv / "
          "epoch_time.csv / training_curve.png")


## 11. Ablation số epoch 2 vs 4 — mục 2.4


In [ ]:
# ============================================================================
# 2.4 ABLATION — chi doi DUNG 1 yeu to: SO EPOCH (2 vs 4)
#
# Doc thang tu checkpoint epoch_2.pth va epoch_4.pth cua chinh lan train nay.
# Cach nay kiem soat chat hon hai lan chay rieng: seed, thu tu du lieu, khoi
# tao deu GIONG HET NHAU vi la cung mot run — dung nghia "chi doi 1 yeu to".
# Do tren rec_val.txt (KHONG dung test), dung ham cham diem chung.
# ============================================================================
ablation_results = []
for ep in ABLATION_EPOCHS:
    w = EPOCH_DIR / f"epoch_{ep}.pth"
    if not w.exists():
        print(f"chua co {w.name} — bo qua (moi train duoc {done_epoch} epoch)")
        continue
    if time_left() < len(val_rows) * SEC_PER_IMG + MODEL_LOAD_S + 300:
        print("khong du ngan sach cho phan con lai cua ablation — bo qua")
        break
    out = RESULTS_DIR / f"ablation_epoch{ep}_val.txt"
    secs, _ = run_process(
        [sys.executable, SCRIPT_DIR / "vietocr_predict.py",
         "--config", CFG_FINETUNE, "--weights", w, "--list", VAL_FILE,
         "--data-root", DATA_DIR, "--out", out],
        f"ablation_ep{ep}.log")
    m, _ = evaluate(VAL_FILE, out)
    tr_s = float(epoch_hist[epoch_hist["epoch"] <= ep]["giay"].sum()) \
        if len(epoch_hist) else None
    ablation_results.append({"so_epoch": ep, **m,
                             "train_seconds": round(tr_s, 1) if tr_s else None,
                             "val_infer_seconds": round(secs, 1)})
    print(f"  epoch {ep}: val acc {m['acc']*100:.2f}%  NED {m['norm_edit_dis']:.4f}"
          f"  | ngan sach con {hms(time_left())}")

if ablation_results:
    ablation_df = pd.DataFrame(ablation_results)
    ablation_df.to_csv(RESULTS_DIR / "ablation_epochs.csv", index=False)
    display(ablation_df)

    best = max(ablation_results, key=lambda r: r["acc"])
    worst = min(ablation_results, key=lambda r: r["acc"])
    d_acc = (best["acc"] - worst["acc"]) * 100
    print(f"\n{best['so_epoch']} epoch thang: val acc {best['acc']*100:.2f}% "
          f"vs {worst['acc']*100:.2f}% cua {worst['so_epoch']} epoch "
          f"(+{d_acc:.2f} diem)\n")
    print(f"""Vi sao, khong chi doc so:
 1. Toan bo lop embedding cho {len(VOCAB) - len(DEFAULT_VOCAB)} ky tu noi them va lop chieu
    dau ra deu duoc KHOI TAO LAI (vocab doi tu {len(DEFAULT_VOCAB)} len {len(VOCAB)}), nen
    chung phai hoc tu con so khong. 2 epoch chua du de nhung tham so nay on dinh.
 2. Anh la ban scan sach cu — nhieu, chu mo, dau thanh chong nhau — khac han
    du lieu tong hop ma vgg_transformer duoc pretrain. Backbone can nhieu buoc
    hon de thich nghi voi phan bo anh nay.
 3. Loi ich giam dan: xem cot val_acc theo tung epoch o bang training_curve
    de biet duong cong da phang chua. Neu tu epoch {worst['so_epoch']} den
    {best['so_epoch']} van doc len ro thi train them nua con an tiep.""")
else:
    print("Khong co checkpoint nao du de lam ablation trong phien nay.")


## 12. Bảng so sánh — mục 2.3


In [ ]:
# ============================================================================
# 2.3 BANG SO SANH — CUNG bo test (rec_test.txt), cung ham cham diem
#     De bai: "Khong duoc so so do tren val voi so do tren test."
# ============================================================================
FINAL_TXT   = RESULTS_DIR / "finetune_test.txt"
FINAL_JSONL = RESULTS_DIR / "pred_test_finetune.jsonl"

final_test_metric, final_recs, final_test_seconds = None, [], 0.0
if FINAL_WEIGHTS is None:
    print("Chua co trong so fine-tune — bo qua bang so sanh.")
else:
    final_test_seconds, _ = run_process(
        [sys.executable, SCRIPT_DIR / "vietocr_predict.py",
         "--config", CFG_FINETUNE, "--weights", FINAL_WEIGHTS,
         "--list", TEST_FILE, "--data-root", DATA_DIR, "--out", FINAL_TXT],
        "finetune_test.log")
    final_test_metric, final_recs = evaluate(TEST_FILE, FINAL_TXT, FINAL_JSONL)
    final_test_metric["inference_seconds"] = round(final_test_seconds, 1)

    comp = pd.DataFrame([
        {"Model": f"VietOCR {MODEL_NAME} goc (chua fine-tune)",
         "acc": round(baseline_metric["acc"], 4),
         "norm_edit_dis": round(baseline_metric["norm_edit_dis"], 4),
         "Thoi gian train": "0"},
        {"Model": f"Fine-tune cua nhom ({done_epoch} epoch)",
         "acc": round(final_test_metric["acc"], 4),
         "norm_edit_dis": round(final_test_metric["norm_edit_dis"], 4),
         "Thoi gian train": hms(train_seconds)},
    ])
    comp.to_csv(RESULTS_DIR / "comparison_test.csv", index=False)
    print(f"Do tren rec_test.txt (n = {final_test_metric['n']}), "
          f"ignore_space = {IGNORE_SPACE}, seed = {SEED}")
    display(comp)

    # kiem tra cheo acc <-> NED: lech nhieu = loi bi don cuc, khong rai deu
    L = int(np.median([len(t) for _, t in test_rows]))
    a_, n_ = final_test_metric["acc"], final_test_metric["norm_edit_dis"]
    if 0 < a_ < 1:
        implied = a_ ** (1 / L)
        print(f"\nNED ma acc du doan (dong dai trung vi {L}): {implied:.4f}")
        print(f"NED that                                : {n_:.4f}")
        if implied - n_ > 0.005:
            print(f"chenh {implied-n_:+.4f} -> loi bi don cuc o mot so dong hong nang;\n"
                  "  xem bang phan loai o cell duoi de biet don o dau")
        else:
            print("hai so khop nhau -> loi rai kha deu tren cac dong")
print("ngan sach con", hms(time_left()))


## 13. Phân tích lỗi — mục 2.5


In [ ]:
# ============================================================================
# 2.5 PHAN TICH LOI — 20 dong sai, 3 loai theo dung ten de bai dat
# ============================================================================
import random

# Dung ma escape chu khong go truc tiep: ky tu to hop la VO HINH trong source,
# rat de bi mat khi copy/sua file.
TONE = {"\u0300", "\u0301", "\u0303", "\u0309", "\u0323"}   # huyen sac nga hoi nang

def strip_tone(s):
    """Bo 5 dau thanh, GIU dau mu/moc (e^ o+ a( ...)."""
    d = unicodedata.normalize("NFD", s)
    return unicodedata.normalize("NFC", "".join(c for c in d if c not in TONE))

def deaccent(s):
    """Ve chu cai La-tinh tran: u+ -> u, e^' -> e, d- -> d."""
    s = s.replace("đ", "d").replace("Đ", "D")
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if not unicodedata.combining(c))

def classify(gt, pred):
    if gt != pred and strip_tone(gt) == strip_tone(pred):
        return "sai dau thanh"
    for op, i1, i2, j1, j2 in Levenshtein.opcodes(gt, pred):
        if op == "insert":
            # so tren chu cai tran: "nguoi" -> "nguuoi" van la lap du u va u+
            # khac codepoint
            seg = deaccent(pred[j1:j2])
            left = deaccent(pred[j1 - 1]) if j1 > 0 else ""
            right = deaccent(pred[j2]) if j2 < len(pred) else ""
            if seg and all(c == left or c == right for c in seg):
                return "lap ky tu"
    return "sai chu cai"

if not final_recs:
    print("Chua co du doan tren test — bo qua phan tich loi.")
else:
    wrong = [r for r in final_recs if metric_text(r["gt"]) != metric_text(r["pred"])]
    sample20 = random.Random(SEED).sample(wrong, min(20, len(wrong)))
    for r in sample20:
        r["loai_loi"] = classify(r["gt"], r["pred"])

    error_df = pd.DataFrame(sample20)[["image", "gt", "pred", "loai_loi"]]
    error_df.to_csv(RESULTS_DIR / "error_analysis_20.csv", index=False)
    pd.set_option("display.max_colwidth", 70)
    display(error_df)

    pct = (error_df["loai_loi"].value_counts(normalize=True) * 100).round(1)
    print(f"Tren 20 mau (de bai yeu cau):")
    for k in ["sai dau thanh", "sai chu cai", "lap ky tu"]:
        print(f"  {k:<16}: {pct.get(k, 0.0):5.1f} %")

    allc = (pd.Series([classify(r["gt"], r["pred"]) for r in wrong])
            .value_counts(normalize=True) * 100)
    print(f"\nTren toan bo {len(wrong)} dong sai (chac chan hon 20 mau):")
    for k in ["sai dau thanh", "sai chu cai", "lap ky tu"]:
        print(f"  {k:<16}: {allc.get(k, 0.0):5.1f} %")

    # --- so voi PaddleOCR: nhom CUT DUOI phai bien mat vi decoder tu hoi quy
    #     khong co rang buoc T >= L nhu CTC ---
    trunc = [r for r in wrong
             if metric_text(r["gt"]).startswith(metric_text(r["pred"]))
             and len(metric_text(r["gt"])) - len(metric_text(r["pred"])) >= 3]
    print(f"\nDong bi CUT DUOI (pred la tien to cua gt, thieu >= 3 ky tu): "
          f"{len(trunc)}/{len(wrong)} = {len(trunc)/len(wrong)*100:.1f}%")
    print("  PaddleOCR co nhom nay vi CTC doi T >= L. Neu o day van cao thi gia\n"
          "  thuyet ve CTC la sai va nguyen nhan nam o cho khac.")

    # --- seq2seq co the lap vo han -> pred dai bat thuong ---
    runaway = [r for r in wrong
               if len(metric_text(r["pred"])) > len(metric_text(r["gt"])) * 1.5 + 10]
    print(f"\nDong pred DAI BAT THUONG (>1.5x nhan, dau hieu decoder lap): {len(runaway)}")
    for r in runaway[:2]:
        print("   gt  :", r["gt"][:70]); print("   pred:", r["pred"][:70])

    # --- tach rieng: sai DAU MU (a^ e^ o^ o+ u+) khac voi sai DAU THANH.
    #     De bai chi liet ke "sai dau thanh", nen e^ -> e duoc xep vao "sai chu cai"
    #     (e va e^ la hai chu khac nhau trong bang chu cai tieng Viet). Dem rieng
    #     o day de bao cao noi duoc chinh xac hon.
    only_hat = sum(1 for r in wrong
                   if strip_tone(metric_text(r["gt"])) != strip_tone(metric_text(r["pred"]))
                   and deaccent(metric_text(r["gt"])) == deaccent(metric_text(r["pred"])))
    print(f"\nTrong so do, {only_hat} dong sai DAU MU/MOC (e^ -> e, o+ -> o, CHUONG -> CHUONG)")

    # --- chi sai dau phu ---
    only_dia = sum(1 for r in wrong
                   if deaccent(metric_text(r["gt"])) == deaccent(metric_text(r["pred"])))
    print(f"\n{only_dia}/{len(wrong)} dong ({only_dia/len(wrong)*100:.1f}%) chi sai o DAU PHU "
          f"(thanh/mu/moc), chu cai tran hoan toan dung.")

    # --- nhan hong: gt chua ky tu ngoai vi_dict.txt ---
    bad_gt = [r for r in wrong if (set(r["gt"]) - {" "}) - dict_chars]
    print(f"\n{len(bad_gt)}/{len(wrong)} dong ({len(bad_gt)/len(wrong)*100:.1f}%) co NHAN chua "
          f"ky tu ngoai vi_dict.txt -> nhan hong, khong phai model sai.")
    for r in bad_gt[:2]:
        print("   gt  :", r["gt"][:70]); print("   pred:", r["pred"][:70])


## 14. Sản phẩm nộp — mục 4


In [ ]:
# ============================================================================
# MUC 4 — SAN PHAM NOP: eval.py, README.md, requirements.txt, results/
# ============================================================================
import platform

REPO = WORK / "repo"; (REPO / "results").mkdir(parents=True, exist_ok=True)
(REPO / "logs").mkdir(exist_ok=True); (REPO / "scripts").mkdir(exist_ok=True)

(REPO / "eval.py").write_text('''"""
Cham diem OCR: acc (exact match) va norm_edit_dis.

    python eval.py results/pred_test_finetune.jsonl
    python eval.py results/pred_test_finetune.jsonl --keep-space
"""
import argparse, json, unicodedata
import numpy as np
from rapidfuzz.distance import Levenshtein

ap = argparse.ArgumentParser()
ap.add_argument("jsonl", help='moi dong: {"image":..,"gt":..,"pred":..}')
ap.add_argument("--keep-space", action="store_true",
                help="mac dinh bo space khi so sanh (ignore_space=True)")
a = ap.parse_args()

def norm(s):
    s = unicodedata.normalize("NFC", s)
    return s if a.keep_space else s.replace(" ", "")

gts, preds = [], []
with open(a.jsonl, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            r = json.loads(line)
            gts.append(norm(r["gt"])); preds.append(norm(r["pred"]))

acc = np.mean([g == p for g, p in zip(gts, preds)])
ned = 1 - np.mean([Levenshtein.normalized_distance(g, p) for g, p in zip(gts, preds)])
cer = sum(Levenshtein.distance(g, p) for g, p in zip(gts, preds)) / sum(max(len(g), 1) for g in gts)
print(f"n              : {len(gts)}")
print(f"ignore_space   : {not a.keep_space}")
print(f"acc            : {acc:.4f}")
print(f"norm_edit_dis  : {ned:.4f}")
print(f"CER            : {cer:.4f}")
''', encoding="utf-8")

import importlib.metadata as _md
def _ver(p):
    try:
        return _md.version(p)
    except Exception:
        return "?"

(REPO / "requirements.txt").write_text(
    f"vietocr=={_ver('vietocr')}\ntorch=={_ver('torch')}\ntorchvision\n"
    "numpy\npandas\npyyaml\nrapidfuzz\nPillow\nlmdb\ntqdm\nalbumentations\nmatplotlib\n",
    encoding="utf-8")

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
summary = {
    "seed": SEED,
    "framework": {"vietocr": _ver("vietocr"), "torch": _ver("torch"), "model": MODEL_NAME},
    "vocab": {"mac_dinh": len(DEFAULT_VOCAB), "dung": len(VOCAB),
              "noi_them": len(VOCAB) - len(DEFAULT_VOCAB)},
    "hyperparams": {"image_height": IMG_HEIGHT, "image_max_width": IMG_MAX_WIDTH,
                    "cnn_ss": CNN_SS, "batch_size": BATCH_SIZE, "max_lr": MAX_LR,
                    "iters_per_epoch": ITERS_PER_EPOCH,
                    "epochs_done": done_epoch, "epochs_target": TARGET_EPOCHS,
                    "ignore_space": IGNORE_SPACE},
    "do_thoi_gian": {"sec_per_iter": round(SEC_PER_ITER, 4),
                     "sec_per_epoch": round(SEC_PER_EPOCH, 1),
                     "sec_per_image_infer": round(SEC_PER_IMG, 4),
                     "fit_epochs_phien_nay": FIT_EPOCHS},
    "ablation_so_epoch": ablation_results,
    "baseline_test": baseline_metric,
    "final_test": final_test_metric,
    "train_seconds": round(train_seconds, 1),
    "training_curve": (eval_hist[["epoch", "iter", "acc_full_seq", "acc_per_char",
                                  "elapsed_s"]].to_dict("records")
                       if len(eval_hist) else []),
    "machine": {"platform": platform.platform(), "python": platform.python_version(),
                "gpu": gpu},
}
json.dump(summary, open(RESULTS_DIR / "summary.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=str)

_ft = final_test_metric or {"acc": float("nan"), "norm_edit_dis": float("nan"), "n": 0}
(REPO / "README.md").write_text(f"""# Nhan dang chu tieng Viet — VietOCR fine-tune

| Model | acc | norm_edit_dis | Thoi gian train |
|---|---|---|---|
| VietOCR `{MODEL_NAME}` goc (chua fine-tune) | {baseline_metric['acc']:.4f} | {baseline_metric['norm_edit_dis']:.4f} | 0 |
| Fine-tune cua nhom ({done_epoch} epoch) | {_ft['acc']:.4f} | {_ft['norm_edit_dis']:.4f} | {hms(train_seconds)} |

Do tren `rec_test.txt` (n = {_ft['n']}), `ignore_space = {IGNORE_SPACE}`, seed = {SEED}.
Ca hai dong do bang cung mot ham trong `eval.py`.

## Moi truong
- VietOCR {_ver('vietocr')}, PyTorch {_ver('torch')}
- {gpu or 'GPU'} — {platform.platform()}
- `pip install -r requirements.txt`

## Vi sao doi tu PaddleOCR sang VietOCR
Ban PaddleOCR (PP-OCRv5 mobile rec, CTC) hoi tu o `acc = 0.606`. Phan tich loi cho thay
phan lon la sai DUNG 1 ky tu dau, va mot nhom dong bi cut duoi. Ca hai deu la diem yeu
co huu cua CTC + greedy decode: giai ma tung frame doc lap khong co ngu canh, va rang
buoc `T >= L` khien nhan dai bi cat. VietOCR dung decoder tu hoi quy — moi ky tu sinh ra
co dieu kien tren cac ky tu truoc — nen co mo hinh ngon ngu ngam, va khong co rang buoc
do dai nao.

## Hai sua doi so voi cau hinh VietOCR mac dinh
1. **`image_height` 32 -> {IMG_HEIGHT}.** O h=32, net gach ngang cua `d` va dau mu chi day
   1-2 pixel nen hay bi mat (`2 dong` doc thanh `2 dong`, `CHUONG` mat dau).
   Kem theo phai sua `cnn.ss`/`cnn.ks` hang cuoi tu `[1,1]` thanh `[2,1]` de tich stride
   chieu cao thanh 32: nho vay feature height van bang 2 va chuoi encoder KHONG dai them,
   trong khi conv duoc nhin anh o do phan giai doc gap doi. Lop pooling khong co tham so
   nen doi `ss`/`ks` khong lam lech trong so pretrained.
2. **`image_max_width` 512 -> {IMG_MAX_WIDTH}.** Ty le w/h trung vi cua bo du lieu la 19.7;
   o chieu cao {IMG_HEIGHT} thi anh trung vi can ~{int(19.7*IMG_HEIGHT)}px. De 512 la nen nat gan het anh.

## Vocab
Vocab mac dinh cua VietOCR ({len(DEFAULT_VOCAB)} ky tu) da du chu tieng Viet co dau nhung
thieu dau cau cua bo nay. Da NOI THEM {len(VOCAB) - len(DEFAULT_VOCAB)} ky tu VAO CUOI
(khong sap xep lai) de chi so 0..{len(DEFAULT_VOCAB)-1} giu nguyen y nghia — chi cac hang
embedding moi la khoi tao lai. Tong: {len(VOCAB)} ky tu.

## Cach chay lai
1. Add `vi_rec_100k.zip` lam Kaggle Dataset (hoac de script tu gdown).
2. Notebook settings: Internet = On, Accelerator = **P100** (VietOCR chi dung 1 GPU).
3. Run All. Cell hieu chuan do toc do that roi tu chon so epoch vua ngan sach, nen
   notebook luon chay het moi cell trong 1 phien.
4. Chua du {TARGET_EPOCHS} epoch: Save Version -> notebook moi -> Add Data chon output
   lan truoc -> Run All. Script tu tim `vietocr_checkpoint.pth` va train tiep.
5. `python eval.py results/pred_test_finetune.jsonl`

## Noi dung
- `results/pred_test_baseline.jsonl`, `results/pred_test_finetune.jsonl`
- `results/comparison_test.csv` — muc 2.3
- `results/ablation_epochs.csv` — muc 2.4 (so epoch 2 vs 4)
- `results/error_analysis_20.csv` — muc 2.5
- `results/training_curve.csv` + `.png`, `per_epoch.csv`, `epoch_time.csv`, `train_steps.csv`
- `results/summary.json` — toan bo cau hinh, phien ban, so do thoi gian
- `scripts/` — script train va suy luan chay qua subprocess
- Checkpoint: (dien link Google Drive / HuggingFace sau khi tai ve)

## Bang phan cong
| Thanh vien | Cong viec | Dong gop |
|---|---|---|
| | | |
""", encoding="utf-8")

for f in RESULTS_DIR.glob("*"):
    if f.suffix in {".csv", ".json", ".jsonl", ".png"}:
        shutil.copy(f, REPO / "results" / f.name)
for f in list(LOG_DIR.glob("*.log")) + list(HIST_DIR.glob("train_*.log")):
    shutil.copy(f, REPO / "logs" / f.name)
for f in SCRIPT_DIR.glob("*.py"):
    shutil.copy(f, REPO / "scripts" / f.name)
for f in CONFIG_DIR.glob("*.yml"):
    shutil.copy(f, REPO / f.name)

bundle = shutil.make_archive(str(WORK / "nop_bai"), "zip", root_dir=REPO)
print("Repo   :", REPO)
print("Bundle :", bundle, f"({os.path.getsize(bundle)/1e6:.1f} MB)")
if FINAL_WEIGHTS:
    print("Ckpt   :", FINAL_WEIGHTS, f"({FINAL_WEIGHTS.stat().st_size/1e6:.1f} MB)")
print("\nTai ve tu tab Output cua Kaggle, roi up checkpoint len Drive/HuggingFace.")
print("Tong thoi gian phien:", hms(time.time() - SESSION_START))
